# DSI Skeleton — Squarepoint Capital

**Автор:** Sergey Nefedov  
**Назначение:** Универсальный шаблон для Data Science Interview (DSI)  
**Использование:** Копируй блоки по мере необходимости, адаптируй под конкретный датасет

---

## Структура ноутбука

| # | Блок | Что делает |
|---|------|------------|
| 1 | Imports & Config | Все библиотеки, настройки отображения |
| 2 | Load & First Look | Загрузка, базовая инспекция |
| 3 | Data Quality | Пропуски, типы, дубликаты, выбросы |
| 4 | EDA — Univariate | Распределения каждой переменной |
| 5 | EDA — Bivariate | Связи между переменными |
| 6 | Feature Engineering | Создание новых признаков |
| 7 | Preprocessing | Скейлинг, кодирование, сплит |
| 8 | Baseline Model | Простая модель как точка отсчёта |
| 9 | ML Models | Ridge, LightGBM, XGBoost |
| 10 | Validation | Walk-forward, CV, метрики |
| 11 | Quant Analysis | IC, ICIR, квинтили (если нужно) |
| 12 | Conclusions | Выводы, следующие шаги |

---
## 1. Imports & Configuration

Загружаем всё сразу в начале — так интервьюер видит, что ты знаешь стек.  
Настраиваем отображение pandas и matplotlib один раз глобально.

In [ ]:
# === CORE ===
import numpy as np
import pandas as pd

# === VISUALIZATION ===
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# === STATISTICS ===
from scipy import stats
from scipy.stats import spearmanr, pearsonr, normaltest, shapiro

# === SKLEARN — PREPROCESSING ===
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# === SKLEARN — MODELS ===
from sklearn.linear_model import Ridge, Lasso, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier

# === SKLEARN — VALIDATION ===
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    TimeSeriesSplit, KFold, GridSearchCV
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix
)

# === BOOSTING ===
import lightgbm as lgb
import xgboost as xgb

# === WARNINGS ===
import warnings
warnings.filterwarnings('ignore')

# === DISPLAY SETTINGS ===
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 1000)

# === PLOT SETTINGS ===
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11
sns.set_palette('muted')

# === RANDOM SEED ===
SEED = 42
np.random.seed(SEED)

print('All imports OK')

---
## 2. Load & First Look

**Цель первых 5 минут:** понять структуру данных, не делая ещё никаких выводов.  
Задаём себе вопросы:
- Сколько строк и колонок?
- Какие типы данных?
- Есть ли очевидные проблемы сразу?
- Что является таргетом (целевой переменной)?

In [ ]:
# --- Загрузка ---
# Адаптируй путь и формат под конкретный файл
df = pd.read_csv('data.csv')          # CSV
# df = pd.read_excel('data.xlsx')     # Excel
# df = pd.read_parquet('data.parquet') # Parquet

# --- Размер ---
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# --- Первый взгляд ---
df.head(10)

In [ ]:
# --- Типы данных и non-null counts ---
# Ищем: object-колонки (возможно категории), float64 vs int64, datetime
df.info()

In [ ]:
# --- Базовая статистика ---
# Смотрим: min/max на адекватность, mean vs median (скошенность), std
df.describe(include='all').T

In [ ]:
# --- Имена колонок ---
print('Columns:')
for i, col in enumerate(df.columns):
    print(f'  {i:2d}. {col} ({df[col].dtype})')

---
## 3. Data Quality

**Никогда не пропускай этот блок.** Грязные данные — главная причина плохих моделей.  
Проверяем четыре вещи:
1. **Пропуски** — сколько, где, паттерн случайный или систематический?
2. **Дубликаты** — одинаковые строки?
3. **Типы** — корректны ли типы данных?
4. **Выбросы** — аномальные значения?

In [ ]:
# === ПРОПУСКИ ===
missing = pd.DataFrame({
    'count':   df.isnull().sum(),
    'pct':     df.isnull().mean() * 100
}).query('count > 0').sort_values('pct', ascending=False)

print(f'Колонок с пропусками: {len(missing)} из {df.shape[1]}')
print(missing.to_string())

# Визуализация паттерна пропусков
if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing['pct'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('% пропусков')
    ax.set_title('Пропуски по колонкам')
    plt.tight_layout()
    plt.show()

In [ ]:
# === ДУБЛИКАТЫ ===
n_dups = df.duplicated().sum()
print(f'Полных дубликатов: {n_dups} ({n_dups/len(df)*100:.2f}%)')

# Если есть ID-колонка — проверяем уникальность
# id_col = 'id'
# print(f'Уникальных {id_col}: {df[id_col].nunique()} из {len(df)}')

In [ ]:
# === ВЫБРОСЫ — IQR метод ===
# Для числовых колонок считаем, сколько точек за 3*IQR
num_cols = df.select_dtypes(include=np.number).columns

outlier_report = []
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 3*IQR, Q3 + 3*IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    if n_out > 0:
        outlier_report.append({'column': col, 'outliers': n_out,
                                'pct': n_out/len(df)*100,
                                'lower': lower, 'upper': upper})

outlier_df = pd.DataFrame(outlier_report).sort_values('pct', ascending=False)
print('Выбросы (3×IQR):')
print(outlier_df.to_string(index=False))

In [ ]:
# === РАСПРЕДЕЛЕНИЕ ТАРГЕТА ===
# Определи таргет — это самое важное
TARGET = 'target'   # <-- ЗАМЕНИ НА РЕАЛЬНОЕ ИМЯ

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма
df[TARGET].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Распределение {TARGET}')
axes[0].axvline(df[TARGET].mean(),   color='red',    linestyle='--', label=f'Mean={df[TARGET].mean():.2f}')
axes[0].axvline(df[TARGET].median(), color='orange', linestyle='--', label=f'Median={df[TARGET].median():.2f}')
axes[0].legend()

# Boxplot
df.boxplot(column=TARGET, ax=axes[1])
axes[1].set_title(f'Boxplot {TARGET}')

plt.tight_layout()
plt.show()

# Числовая сводка
print(df[TARGET].describe())
print(f'Skewness: {df[TARGET].skew():.3f}')   # >1 или <-1 = сильный скос
print(f'Kurtosis: {df[TARGET].kurt():.3f}')   # >3 = тяжёлые хвосты

---
## 4. EDA — Univariate Analysis

Смотрим на каждую переменную отдельно.  
**Числовые:** распределение, скошенность, выбросы.  
**Категориальные:** количество уникальных значений, частоты, редкие категории.

In [ ]:
# === ЧИСЛОВЫЕ ПЕРЕМЕННЫЕ ===
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f'Числовых колонок: {len(num_cols)}')

# Сетка гистограмм
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col}\nskew={df[col].skew():.2f}', fontsize=9)
    axes[i].tick_params(labelsize=8)

# Скрываем лишние сабплоты
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Распределения числовых переменных', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === КАТЕГОРИАЛЬНЫЕ ПЕРЕМЕННЫЕ ===
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Категориальных колонок: {len(cat_cols)}')

for col in cat_cols:
    n_unique = df[col].nunique()
    top_vals = df[col].value_counts().head(10)
    print(f'\n{col}: {n_unique} уникальных значений')
    print(top_vals.to_string())

    if n_unique <= 20:
        fig, ax = plt.subplots(figsize=(8, 3))
        top_vals.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title(f'{col} — частоты')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

---
## 5. EDA — Bivariate Analysis

Смотрим на связи между переменными и таргетом.  
Ключевые вопросы:
- Какие признаки сильнее всего коррелируют с таргетом?
- Есть ли мультиколлинеарность между признаками?
- Есть ли нелинейные зависимости?

In [ ]:
# === CORRELATION MATRIX ===
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # верхний треугольник скрываем
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix (Pearson)')
plt.tight_layout()
plt.show()

In [ ]:
# === КОРРЕЛЯЦИЯ С ТАРГЕТОМ ===
# Pearson (линейная) и Spearman (монотонная) — сравниваем оба
target_corr = pd.DataFrame({
    'pearson':  [df[col].corr(df[TARGET], method='pearson')  for col in num_cols],
    'spearman': [df[col].corr(df[TARGET], method='spearman') for col in num_cols]
}, index=num_cols).drop(TARGET, errors='ignore')

target_corr['abs_spearman'] = target_corr['spearman'].abs()
target_corr = target_corr.sort_values('abs_spearman', ascending=False)

print('Корреляция признаков с таргетом:')
print(target_corr.drop('abs_spearman', axis=1).to_string())

# График
fig, ax = plt.subplots(figsize=(10, 6))
target_corr['spearman'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Spearman корреляция с {TARGET}')
plt.tight_layout()
plt.show()

In [ ]:
# === SCATTER PLOTS топ признаков с таргетом ===
top_features = target_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df[TARGET], alpha=0.3, s=10, color='steelblue')
    # Линия тренда
    mask_valid = df[[col, TARGET]].dropna()
    z = np.polyfit(mask_valid[col], mask_valid[TARGET], 1)
    p = np.poly1d(z)
    x_line = np.linspace(mask_valid[col].min(), mask_valid[col].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=1.5)
    r = df[col].corr(df[TARGET], method='spearman')
    axes[i].set_title(f'{col}\nSpearman ρ = {r:.3f}', fontsize=9)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)

plt.suptitle('Топ признаки vs таргет', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Feature Engineering

**Создание новых признаков — часто важнее выбора модели.**  
Типичные трансформации:
- Логарифм для скошенных распределений
- Взаимодействия между признаками
- Временные признаки из дат
- Ранговые трансформации (важно для quant!)

> Важно: все трансформации делаем ПОСЛЕ train/test split или внутри Pipeline, чтобы избежать data leakage.

In [ ]:
df_fe = df.copy()

# === ЛОГ-ТРАНСФОРМАЦИЯ скошенных признаков ===
# Применяем если skew > 1 и все значения > 0
skewed_cols = [col for col in num_cols
               if df[col].skew() > 1 and (df[col] > 0).all()]
print(f'Скошенные колонки для log-transform: {skewed_cols}')

for col in skewed_cols:
    df_fe[f'log_{col}'] = np.log1p(df_fe[col])  # log1p = log(1+x), безопасно для 0

# === РАНГОВАЯ ТРАНСФОРМАЦИЯ (quant-стиль) ===
# Нормируем ранги в [-0.5, 0.5] — стандарт в факторных моделях
def rank_normalize(series):
    """Ранговая нормализация в [-0.5, 0.5]"""
    ranks = series.rank(method='average', na_option='keep')
    return (ranks - 1) / (ranks.count() - 1) - 0.5

for col in num_cols:
    if col != TARGET:
        df_fe[f'rank_{col}'] = rank_normalize(df_fe[col])

# === ВРЕМЕННЫЕ ПРИЗНАКИ (если есть дата) ===
# date_col = 'date'
# df_fe[date_col] = pd.to_datetime(df_fe[date_col])
# df_fe['year']        = df_fe[date_col].dt.year
# df_fe['month']       = df_fe[date_col].dt.month
# df_fe['dayofweek']   = df_fe[date_col].dt.dayofweek
# df_fe['is_month_end']= df_fe[date_col].dt.is_month_end.astype(int)

# === ВЗАИМОДЕЙСТВИЯ (пример) ===
# df_fe['feat1_x_feat2'] = df_fe['feat1'] * df_fe['feat2']
# df_fe['feat1_ratio_feat2'] = df_fe['feat1'] / (df_fe['feat2'] + 1e-8)

print(f'Shape после Feature Engineering: {df_fe.shape}')

---
## 7. Preprocessing & Train/Test Split

**Золотое правило:** test set не должен влиять на обучение ни в каком виде.  
Для временных рядов — только временной сплит (не random!).

Pipeline гарантирует, что fit происходит только на train данных.

In [ ]:
# === ОПРЕДЕЛЯЕМ ПРИЗНАКИ И ТАРГЕТ ===
FEATURES = [col for col in df_fe.columns
            if col != TARGET and df_fe[col].dtype in [np.float64, np.int64]]

X = df_fe[FEATURES]
y = df_fe[TARGET]

print(f'Признаков: {len(FEATURES)}')
print(f'Таргет: {TARGET}')

In [ ]:
# === СПЛИТ ===

# --- Вариант A: случайный (для iid данных) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# --- Вариант B: временной (для временных рядов) ---
# split_idx = int(len(X) * 0.8)
# X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
# y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

In [ ]:
# === PREPROCESSING PIPELINE ===
# Числовые: заполняем медианой, скейлим
# Категориальные: заполняем модой, OneHot

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
], remainder='drop')

# Применяем только fit на train!
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)     # только transform!

print(f'Processed shape: {X_train_proc.shape}')

---
## 8. Baseline Model

**Всегда начинай с простейшей модели.** Это даёт точку отсчёта.  
Если сложная модель не бьёт baseline — что-то не так с данными или пайплайном.

Для регрессии: предсказываем среднее (DummyRegressor).  
Для классификации: предсказываем самый частый класс (DummyClassifier).

In [ ]:
def evaluate_regression(model, X_tr, y_tr, X_te, y_te, name='Model'):
    """Обучает и оценивает регрессионную модель. Возвращает dict с метриками."""
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    metrics = {
        'model':    name,
        'RMSE_train': np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        'RMSE_test':  np.sqrt(mean_squared_error(y_te, y_pred_te)),
        'MAE_test':   mean_absolute_error(y_te, y_pred_te),
        'R2_train':   r2_score(y_tr, y_pred_tr),
        'R2_test':    r2_score(y_te, y_pred_te),
    }

    # Разница train/test RMSE — индикатор overfitting
    metrics['overfit_gap'] = metrics['RMSE_test'] - metrics['RMSE_train']

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  RMSE  train: {metrics['RMSE_train']:.4f}")
    print(f"  RMSE  test:  {metrics['RMSE_test']:.4f}  (gap: {metrics['overfit_gap']:+.4f})")
    print(f"  MAE   test:  {metrics['MAE_test']:.4f}")
    print(f"  R²    train: {metrics['R2_train']:.4f}")
    print(f"  R²    test:  {metrics['R2_test']:.4f}")

    return metrics


# --- Baseline ---
baseline = DummyRegressor(strategy='mean')
results = []
results.append(evaluate_regression(
    baseline, X_train_proc, y_train, X_test_proc, y_test, 'Baseline (mean)'
))

---
## 9. ML Models

Порядок усложнения: Linear → Ridge → LightGBM → XGBoost.  
На DSI важно объяснить почему выбрал именно эту модель и что означают параметры.

In [ ]:
# === RIDGE REGRESSION ===
# Хорошо при мультиколлинеарности, когда признаков много
# alpha — сила регуляризации (больше alpha = сильнее сжатие коэффициентов)
ridge = Ridge(alpha=1.0, random_state=SEED)
results.append(evaluate_regression(
    ridge, X_train_proc, y_train, X_test_proc, y_test, 'Ridge'
))

In [ ]:
# === RIDGE — визуализация коэффициентов ===
# Важно: показывает какие признаки и в каком направлении влияют на таргет
try:
    coef_df = pd.DataFrame({
        'feature': num_features,
        'coef':    ridge.coef_[:len(num_features)]
    }).sort_values('coef', key=abs, ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['coef']]
    ax.barh(coef_df['feature'], coef_df['coef'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Ridge — топ-20 коэффициентов')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Не удалось построить коэффициенты: {e}')

In [ ]:
# === LIGHTGBM ===
# Быстрый, хорошо работает с большими данными и категориями
# n_estimators — кол-во деревьев
# learning_rate — шаг обучения (меньше = лучше, но медленнее)
# num_leaves — сложность каждого дерева (32-128 обычно)
# min_child_samples — min наблюдений в листе, контролирует overfitting

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    verbose=-1
)

results.append(evaluate_regression(
    lgb_model, X_train_proc, y_train, X_test_proc, y_test, 'LightGBM'
))

In [ ]:
# === XGBOOST ===
# Более консервативный, лучше с шумными данными
# max_depth — глубина дерева (3-6 обычно)
# reg_alpha, reg_lambda — L1/L2 регуляризация

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    verbosity=0
)

results.append(evaluate_regression(
    xgb_model, X_train_proc, y_train, X_test_proc, y_test, 'XGBoost'
))

In [ ]:
# === СРАВНИТЕЛЬНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ ===
results_df = pd.DataFrame(results).set_index('model')
print('\nСравнение моделей:')
print(results_df.to_string())

# График RMSE
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['RMSE_train'], width, label='Train', color='steelblue', alpha=0.8)
ax.bar(x + width/2, results_df['RMSE_test'],  width, label='Test',  color='coral',     alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=20)
ax.set_ylabel('RMSE')
ax.set_title('Train vs Test RMSE по моделям')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === FEATURE IMPORTANCE — LightGBM ===
# Показывает какие признаки модель считает важными
# gain — важность по вкладу в снижение ошибки (предпочтительно)
# split — важность по количеству использований в разбиениях

fi = pd.DataFrame({
    'feature':   num_features[:X_train_proc.shape[1]],
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi['feature'], fi['importance'], color='steelblue')
ax.set_title('LightGBM — Feature Importance (top 20)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
### 9.5 SHAP Values — почему модель так предсказывает

Feature importance показывает **какие** признаки важны.  
SHAP показывает **как именно** каждый признак влияет на конкретное предсказание — в каком направлении, при каких значениях.

**Зачем нужно:**
- На DSI после feature importance логичный следующий шаг — «а в какую сторону влияет?» SHAP отвечает на это
- Если интервьюер спросит «почему модель предсказала именно это значение для строки X» — waterfall plot даёт точный ответ
- Помогает находить нелинейные зависимости и взаимодействия фич, которые не видны через простую корреляцию

**Что такое SHAP:**  
Shapley values из теории игр — единственный способ «справедливо» распределить вклад каждого игрока (фичи) в общий выигрыш (предсказание).  
Для каждой строки SHAP раскладывает предсказание на вклады: `prediction = base_value + sum(shap_values_per_feature)`

**Четыре главных типа графиков и как их читать:**

| График | Что показывает | Как читать |
|--------|----------------|------------|
| **Beeswarm (summary)** | Распределение SHAP для каждой фичи | Точка = одно наблюдение. Цвет = значение фичи (красный=высокое, синий=низкое). X = вклад в предсказание |
| **Bar (mean \|SHAP\|)** | Глобальная важность | Высота = средний модуль SHAP. Аналог feature importance, но в единицах таргета |
| **Dependence plot** | Эффект фичи на разных её значениях | X = значение фичи, Y = SHAP. Линия — нелинейность. Цвет — взаимодействие со второй фичей |
| **Waterfall** | Объяснение одного предсказания | От base value к финальному предсказанию: каждая фича двигает вверх (+) или вниз (−) |

**Как интерпретировать beeswarm — самое важное:**
- Красные точки **справа** + синие **слева** → фича **положительно** влияет на таргет (выше значение → выше предсказание)
- Красные **слева** + синие **справа** → **отрицательное** влияние
- Точки **смешаны и близко к 0** → фича не влияет (несмотря на возможную важность по другим метрикам — это сигнал к удалению)
- **Ширина** распределения по горизонтали = насколько сильно влияет

**Реалистичные ожидания:**
- Топ-3 фичи обычно отвечают за 50%+ предсказательной силы
- Если все фичи одинаково «важны» по SHAP — модель плохо обучилась или фичи слишком похожи
- Если direction в SHAP противоположен ожидаемому из доменных знаний — это либо мультиколлинеарность, либо leakage, либо открытие

In [ ]:
# === SHAP — установка и импорт ===
# Если SHAP не установлен:
# !pip install shap

try:
    import shap
    print(f'SHAP version: {shap.__version__}')
except ImportError:
    print('SHAP not installed. Run: pip install shap')
    shap = None

In [ ]:
# === ВЫЧИСЛЕНИЕ SHAP VALUES ===
# Используем простой LightGBM на сырых числовых фичах (с понятными именами).
# Для tree-based моделей SHAP считается ТОЧНО и быстро через TreeExplainer.

# Готовим данные с понятными именами фич (без OHE для простоты)
X_train_shap = X_train[num_features].fillna(X_train[num_features].median())
X_test_shap  = X_test[num_features].fillna(X_train[num_features].median())

# Тренируем LightGBM с исходными именами колонок
lgb_for_shap = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    verbose=-1
)
lgb_for_shap.fit(X_train_shap, y_train)

# Создаём explainer и считаем SHAP values на тестовом множестве
# Для tree-моделей это очень быстро (секунды для тысяч строк)
explainer = shap.TreeExplainer(lgb_for_shap)
shap_values = explainer.shap_values(X_test_shap)

print(f'SHAP values shape: {shap_values.shape}')   # (n_samples, n_features)
print(f'Base value: {explainer.expected_value:.4f}')
print(f'Sum check: prediction = base + sum(shap)')
print(f'  Pred[0]:        {lgb_for_shap.predict(X_test_shap.iloc[[0]])[0]:.4f}')
print(f'  Base + Σshap:   {explainer.expected_value + shap_values[0].sum():.4f}')

In [ ]:
# === BEESWARM SUMMARY PLOT ===
# Самый информативный график SHAP. Покрывает 80% потребностей DSI.
# 
# Как читать:
#   - Каждая строка - одна фича (отсортированы по важности сверху)
#   - Каждая точка - одно наблюдение из X_test
#   - Цвет точки  - значение фичи (красный=высокое, синий=низкое)
#   - Позиция X   - вклад в предсказание (положительный или отрицательный)
# 
# Если красные точки СПРАВА от 0 - фича положительно влияет на таргет.
# Если красные СЛЕВА                - отрицательно.

shap.summary_plot(shap_values, X_test_shap, max_display=15, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# === BAR PLOT — глобальная важность по SHAP ===
# Альтернатива стандартному feature_importance, но в единицах таргета.
# Высота бара = mean(|SHAP|) для каждой фичи.
# Преимущество перед feature_importance: интерпретируемая шкала, нет bias к категориям с большим n_splits.

shap.summary_plot(shap_values, X_test_shap, plot_type='bar', max_display=15, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# === DEPENDENCE PLOT — нелинейности и взаимодействия ===
# Берём топ-1 фичу по SHAP и смотрим, как меняется её вклад в зависимости от значения.
# 
# Как читать:
#   - X: значение фичи
#   - Y: SHAP value (вклад в предсказание)
#   - Если линия монотонна - линейный эффект
#   - Если изгибается  - нелинейность (полезно для feature engineering!)
#   - Цвет точек: вторая фича, выбранная автоматически как самая взаимодействующая

# Топ-1 фича по среднему |SHAP|
top_feature_idx = np.argsort(np.abs(shap_values).mean(axis=0))[::-1][0]
top_feature = X_test_shap.columns[top_feature_idx]
print(f'Dependence plot для: {top_feature}')

shap.dependence_plot(top_feature, shap_values, X_test_shap, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# === WATERFALL PLOT — объяснение одного предсказания ===
# Это ответ на вопрос интервьюера: "почему модель предсказала именно это для строки X?"
# 
# Как читать:
#   - Снизу: base value E[f(X)] - среднее предсказание модели
#   - Каждая стрелка вверх (красная)   = фича повысила предсказание
#   - Каждая стрелка вниз  (синяя)     = фича понизила
#   - Сверху: финальное предсказание f(x)

# Берём первое наблюдение из теста для примера
sample_idx = 0
print(f'Объясняем предсказание для строки {sample_idx}:')
print(f'  Истинное значение: {y_test.iloc[sample_idx]:.4f}')
print(f'  Предсказание:      {lgb_for_shap.predict(X_test_shap.iloc[[sample_idx]])[0]:.4f}')

# Современный API SHAP:
shap_explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_test_shap.iloc[sample_idx].values,
    feature_names=X_test_shap.columns.tolist()
)
shap.waterfall_plot(shap_explanation, max_display=10, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# === ИНТЕРПРЕТАЦИЯ — топ драйверы и направление ===
# Полезная сводная таблица: для каждой топ-фичи показывает её средний |SHAP| и знак среднего эффекта.

shap_summary_df = pd.DataFrame({
    'feature':       X_test_shap.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
    'mean_shap':     shap_values.mean(axis=0),  # средний знак
}).sort_values('mean_abs_shap', ascending=False).head(15)

shap_summary_df['direction'] = shap_summary_df['mean_shap'].apply(
    lambda x: '↑ positive' if x > 0 else '↓ negative'
)

print('Топ-15 фич по SHAP с направлением среднего эффекта:')
print(shap_summary_df.to_string(index=False))

# Note: 'mean_shap' близко к 0 не значит "не важна" - может быть симметричный эффект.
# Смотри beeswarm выше для полной картины.

---
## 10. Validation

### Walk-Forward Validation (для временных рядов)

**Почему не random CV для временных рядов?**  
Random CV нарушает временную структуру — train может содержать данные ПОСЛЕ test.  
Это называется **lookahead bias** и приводит к завышенным метрикам.

Walk-forward имитирует реальный трейдинг: обучаемся на прошлом, тестируем на будущем.

In [ ]:
# === WALK-FORWARD VALIDATION ===
def walk_forward_validation(model, X, y, n_splits=5):
    """
    Walk-forward (expanding window) cross-validation.
    
    Каждый fold:
    - Train: все данные до текущего момента
    - Test:  следующий период
    
    Возвращает список RMSE по каждому fold.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2   = r2_score(y_te, y_pred)
        fold_metrics.append({'fold': fold+1, 'n_train': len(train_idx),
                              'n_test': len(test_idx), 'RMSE': rmse, 'R2': r2})

        print(f'  Fold {fold+1}: train={len(train_idx):5d}, test={len(test_idx):4d} | RMSE={rmse:.4f}, R²={r2:.4f}')

    metrics_df = pd.DataFrame(fold_metrics)
    print(f'\n  Mean RMSE: {metrics_df["RMSE"].mean():.4f} ± {metrics_df["RMSE"].std():.4f}')
    print(f'  Mean R²:   {metrics_df["R2"].mean():.4f} ± {metrics_df["R2"].std():.4f}')
    return metrics_df


print('Walk-Forward Validation — LightGBM:')
wf_results = walk_forward_validation(lgb_model, X_train_proc, y_train, n_splits=5)

---
## 10.5 Confusion Matrix — анатомия ошибок классификации

Если задача DSI — классификация (а не регрессия), confusion matrix обязательна.  
**Accuracy одна сама по себе врёт** на дисбалансе классов: 99% accuracy на 1:99 распределении = модель просто всегда предсказывает мажоритарный класс. Бесполезно.

**Что показывает confusion matrix:**  
Разбивает все ошибки модели по типам — что мы пропустили, что лишнего предсказали.

Для бинарной классификации:

|                  | Predicted: No | Predicted: Yes |
|------------------|---------------|----------------|
| **Actual: No**   | TN            | FP             |
| **Actual: Yes**  | FN            | TP             |

**Производные метрики и что они отвечают:**

| Метрика | Формула | Отвечает на вопрос |
|---------|---------|---------------------|
| **Accuracy** | (TP+TN)/N | Какая доля всех предсказаний правильна? |
| **Precision** | TP/(TP+FP) | Из тех, кого предсказали Yes — сколько действительно Yes? |
| **Recall** (sensitivity, TPR) | TP/(TP+FN) | Из всех настоящих Yes — сколько мы поймали? |
| **Specificity** (TNR) | TN/(TN+FP) | Из всех настоящих No — сколько правильно отвергли? |
| **F1** | 2·P·R / (P+R) | Гармоническое среднее P и R, баланс |

**Главный вопрос на DSI: что важнее в этой задаче — precision или recall?**

- **Recall важнее** когда **дорого пропустить позитив**: детекция мошенничества, диагностика рака, обнаружение сбоев. Лучше ложная тревога, чем пропуск.
- **Precision важнее** когда **дорого ложное срабатывание**: рекомендации (нельзя спамить пользователя), блокировка транзакций (нельзя бесить честных клиентов), таргетированный маркетинг (стоимость касания).
- Если асимметрия не очевидна → **F1 как компромисс**.

**Threshold tuning:**  
Confusion matrix зависит от порога (по умолчанию 0.5 для probability → class). Сдвигая порог:
- Threshold ↑ → меньше FP, но больше FN (выше precision, ниже recall)
- Threshold ↓ → больше FP, меньше FN (выше recall, ниже precision)
- Оптимальный порог зависит от business cost: часто `argmax(F1)` или явный cost matrix

**Какой curve смотреть в зависимости от баланса:**
- Сбалансированные классы → **ROC-AUC** (TPR vs FPR)
- Сильный дисбаланс (1:50+) → **PR-AUC** (Precision vs Recall). ROC-AUC обманчиво высокий на дисбалансе.

In [ ]:
# === ПРИМЕР: создаём binary классификацию из текущих данных ===
# Этот блок применяй когда задача classification.
# Если у тебя уже есть бинарный таргет - просто используй его вместо y_class.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc,
    roc_auc_score, average_precision_score, f1_score
)

# Создаём бинарный таргет: y > median = класс 1
threshold_y = y_train.median()
y_train_clf = (y_train > threshold_y).astype(int)
y_test_clf  = (y_test  > threshold_y).astype(int)

print(f'Class distribution (train):')
print(y_train_clf.value_counts(normalize=True).round(3))
print(f'Class distribution (test):')
print(y_test_clf.value_counts(normalize=True).round(3))

In [ ]:
# === ОБУЧАЕМ КЛАССИФИКАТОР ===
# Logistic Regression - даёт калиброванные вероятности, хороший baseline.

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
clf.fit(X_train_proc, y_train_clf)

# predict_proba возвращает [P(class=0), P(class=1)] для каждой строки
y_proba = clf.predict_proba(X_test_proc)[:, 1]
y_pred  = (y_proba >= 0.5).astype(int)   # стандартный threshold

# Базовые метрики
print(f'Accuracy:  {(y_pred == y_test_clf).mean():.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test_clf, y_proba):.4f}')
print(f'PR-AUC:    {average_precision_score(y_test_clf, y_proba):.4f}')
print(f'F1:        {f1_score(y_test_clf, y_pred):.4f}')

In [ ]:
# === CONFUSION MATRIX — RAW И NORMALIZED ===
# Смотрим обе версии:
#   - Raw:        абсолютные числа — видна реальная масса ошибок
#   - Normalized по rows: class-conditional rates — что модель делает с каждым настоящим классом
#                          (recall лежит на диагонали для positive класса)

cm_raw  = confusion_matrix(y_test_clf, y_pred)
cm_norm = confusion_matrix(y_test_clf, y_pred, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No', 'Pred: Yes'],
            yticklabels=['True: No', 'True: Yes'],
            ax=axes[0], cbar=False, annot_kws={'size': 14})
axes[0].set_title('Confusion Matrix — Raw counts')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalized по rows (class-conditional rates)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Pred: No', 'Pred: Yes'],
            yticklabels=['True: No', 'True: Yes'],
            ax=axes[1], cbar=False, annot_kws={'size': 14})
axes[1].set_title('Confusion Matrix — Normalized by row\n(class-conditional rates)')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

# Распаковываем для удобства
TN, FP, FN, TP = cm_raw.ravel()
print(f'\nDecomposition:')
print(f'  TN (true negative):  {TN:5d}  - правильно сказали No')
print(f'  FP (false positive): {FP:5d}  - ложная тревога (Type I error)')
print(f'  FN (false negative): {FN:5d}  - пропустили (Type II error)')
print(f'  TP (true positive):  {TP:5d}  - правильно сказали Yes')

In [ ]:
# === CLASSIFICATION REPORT + интерпретация ===

print('Classification Report:')
print(classification_report(y_test_clf, y_pred, target_names=['No', 'Yes'], digits=4))

# Ручной расчёт ключевых метрик для прозрачности
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

print(f'Manual check:')
print(f'  Precision   = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.4f}')
print(f'  Recall      = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.4f}')
print(f'  Specificity = TN/(TN+FP) = {TN}/({TN}+{FP}) = {specificity:.4f}')

# Интерпретация для DSI:
# 
# Precision = X% значит: из тех, кому модель предсказала Yes, X% действительно Yes.
#   Низкий precision = много false alarms.
# Recall = Y% значит: из всех настоящих Yes, модель поймала Y%.
#   Низкий recall = много пропущенных кейсов.
# F1 балансирует обе метрики; полезен когда нет явной асимметрии в стоимости ошибок.

In [ ]:
# === ROC И PR КРИВЫЕ ===
# ROC      - универсальная метрика, но обманчиво высокая на дисбалансе.
# PR-curve - чувствительна к дисбалансу, baseline = доля позитивов.

fpr, tpr, roc_thresholds = roc_curve(y_test_clf, y_proba)
precisions, recalls, pr_thresholds = precision_recall_curve(y_test_clf, y_proba)
roc_auc_val = roc_auc_score(y_test_clf, y_proba)
pr_auc_val  = average_precision_score(y_test_clf, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'ROC-AUC = {roc_auc_val:.4f}')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# PR
baseline = y_test_clf.mean()
axes[1].plot(recalls, precisions, color='coral', linewidth=2, label=f'PR-AUC = {pr_auc_val:.4f}')
axes[1].axhline(baseline, color='gray', linestyle='--', label=f'Baseline = {baseline:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Интерпретация: 
# - ROC ниже диагонали = модель хуже случайного. Между диагональю и кривой = модель лучше random.
# - PR baseline = доля позитивов в выборке. Кривая выше = модель лучше случайной.
# - На сильном дисбалансе ROC-AUC может быть 0.95 при PR-AUC 0.30 - PR-AUC честнее.

In [ ]:
# === ПОДБОР THRESHOLD под business need ===
# Если стоимость FP и FN различна - default 0.5 не оптимален.
# Перебираем thresholds, считаем precision/recall/F1.

thresholds_grid = np.linspace(0.05, 0.95, 91)
records = []
for t in thresholds_grid:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test_clf, y_pred_t)
    if cm_t.shape == (2, 2):
        tn, fp, fn, tp = cm_t.ravel()
        prec_t = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec_t  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t) if (prec_t + rec_t) > 0 else 0
        records.append({'threshold': t, 'precision': prec_t,
                        'recall': rec_t, 'f1': f1_t})

th_df = pd.DataFrame(records)

# Optimal F1 threshold
best_f1_idx = th_df['f1'].idxmax()
best_thresh_f1 = th_df.loc[best_f1_idx, 'threshold']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(th_df['threshold'], th_df['precision'], label='Precision', color='steelblue', linewidth=2)
ax.plot(th_df['threshold'], th_df['recall'],    label='Recall',    color='coral',     linewidth=2)
ax.plot(th_df['threshold'], th_df['f1'],        label='F1',        color='darkgreen', linewidth=2)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Default 0.5')
ax.axvline(best_thresh_f1, color='red', linestyle='--', alpha=0.7,
           label=f'Best F1 @ {best_thresh_f1:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Threshold maximizing F1: {best_thresh_f1:.3f}')
print(f'  Precision: {th_df.loc[best_f1_idx, "precision"]:.4f}')
print(f'  Recall:    {th_df.loc[best_f1_idx, "recall"]:.4f}')
print(f'  F1:        {th_df.loc[best_f1_idx, "f1"]:.4f}')

# В реальной задаче меняй objective:
#   - Precision-критичная задача: max(precision) при recall >= X
#   - Recall-критичная задача:    max(recall)    при precision >= Y
#   - Cost-sensitive:             min(C_FP * FP + C_FN * FN)

---
## 11. Quant Analysis — IC / ICIR / Quintiles

Этот блок специфичен для quant research. Применяй когда задача связана с предсказанием доходностей или ранжированием активов.

**IC (Information Coefficient)** — Spearman корреляция между предсказанными значениями и реализованными доходностями.  
**ICIR** — IC / std(IC) — аналог коэффициента Шарпа для факторов.  
**Квинтильный анализ** — разбиваем на 5 групп по фактору, смотрим доходность каждой группы.

In [ ]:
# === IC — Information Coefficient ===

def compute_ic(predictions, returns):
    """
    Вычисляет IC = Spearman корреляция между предсказаниями и реализованными доходностями.
    
    Args:
        predictions: Series — предсказанные значения фактора
        returns:     Series — реализованные доходности
    Returns:
        float — IC от -1 до 1
    """
    ic, pval = spearmanr(predictions, returns, nan_policy='omit')
    return ic, pval


def compute_ic_series(df, factor_col, return_col, date_col):
    """
    Считает IC на каждый период (дату).
    
    Args:
        df:         DataFrame с факторными значениями и доходностями
        factor_col: название колонки с фактором
        return_col: название колонки с доходностью
        date_col:   название колонки с датой/периодом
    Returns:
        Series — IC по каждому периоду
    """
    ic_series = (
        df.groupby(date_col)
          .apply(lambda g: spearmanr(g[factor_col], g[return_col],
                                     nan_policy='omit')[0])
    )
    return ic_series


def compute_icir(ic_series):
    """
    ICIR = mean(IC) / std(IC)
    Аналог Шарпа: награда (средний IC) за единицу риска (волатильность IC).
    """
    mean_ic = ic_series.mean()
    std_ic  = ic_series.std()
    icir    = mean_ic / std_ic if std_ic > 0 else 0

    # t-статистика для проверки значимости
    n = len(ic_series)
    t_stat = icir * np.sqrt(n)
    p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))

    return {
        'mean_IC':  mean_ic,
        'std_IC':   std_ic,
        'ICIR':     icir,
        't_stat':   t_stat,
        'p_value':  p_val,
        'n_periods': n,
        'significant': p_val < 0.05
    }


def plot_ic_analysis(ic_series, title='IC Analysis'):
    """Визуализация IC: временной ряд + гистограмма + кумулятивный IC."""
    stats_dict = compute_icir(ic_series)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. IC по времени
    ic_series.plot(ax=axes[0], color='steelblue', linewidth=1)
    axes[0].axhline(0, color='black', linewidth=0.8)
    axes[0].axhline(ic_series.mean(), color='red', linestyle='--',
                    label=f'Mean={ic_series.mean():.4f}')
    axes[0].fill_between(ic_series.index, ic_series, 0,
                         where=ic_series > 0, color='steelblue', alpha=0.3)
    axes[0].fill_between(ic_series.index, ic_series, 0,
                         where=ic_series < 0, color='coral',     alpha=0.3)
    axes[0].set_title('IC по периодам')
    axes[0].legend(fontsize=9)

    # 2. Гистограмма IC
    axes[1].hist(ic_series, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].axvline(ic_series.mean(), color='red', linestyle='--')
    axes[1].set_title('Распределение IC')
    axes[1].set_xlabel('IC')

    # 3. Кумулятивный IC
    ic_series.cumsum().plot(ax=axes[2], color='steelblue', linewidth=1.5)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].set_title('Кумулятивный IC')

    plt.suptitle(
        f"{title} | Mean IC={stats_dict['mean_IC']:.4f} | "
        f"ICIR={stats_dict['ICIR']:.3f} | t={stats_dict['t_stat']:.2f} | "
        f"p={stats_dict['p_value']:.4f} ({'sig' if stats_dict['significant'] else 'insig'})"
    )
    plt.tight_layout()
    plt.show()

    print('\nIC Statistics:')
    for k, v in stats_dict.items():
        print(f'  {k:12s}: {v}')


print('Функции IC/ICIR загружены. Пример использования:')
print('  ic_series = compute_ic_series(df, factor_col="factor", return_col="return", date_col="date")')
print('  plot_ic_analysis(ic_series)')

In [ ]:
# === QUINTILE ANALYSIS ===

def quintile_analysis(df, factor_col, return_col, date_col=None, n_quantiles=5):
    """
    Квинтильный анализ фактора.
    
    Идея: если фактор работает, то Q1 (самые низкие значения) и Q5 (самые высокие)
    должны иметь систематически разные доходности.
    Spread Q5-Q1 = "alpha" фактора.
    """
    df = df.copy()

    if date_col:
        # Ранжируем внутри каждого периода
        df['quantile'] = df.groupby(date_col)[factor_col].transform(
            lambda x: pd.qcut(x.rank(method='first'), n_quantiles,
                               labels=range(1, n_quantiles+1))
        )
    else:
        df['quantile'] = pd.qcut(df[factor_col].rank(method='first'),
                                  n_quantiles, labels=range(1, n_quantiles+1))

    # Средняя доходность по квинтилям
    quint_returns = df.groupby('quantile')[return_col].agg(['mean', 'std', 'count'])
    quint_returns['sharpe'] = quint_returns['mean'] / quint_returns['std']
    quint_returns.index = [f'Q{i}' for i in quint_returns.index]

    # Spread Q5 - Q1
    spread = quint_returns['mean'].iloc[-1] - quint_returns['mean'].iloc[0]
    print(f'Квинтильный анализ ({factor_col} → {return_col}):')
    print(quint_returns.to_string())
    print(f'\nSpread Q5-Q1: {spread:.4f}')

    # График
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['coral' if r < 0 else 'steelblue' for r in quint_returns['mean']]
    ax.bar(quint_returns.index, quint_returns['mean'], color=colors, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Средняя доходность по квинтилям ({factor_col})')
    ax.set_ylabel('Средняя доходность')
    ax.set_xlabel('Квинтиль (Q1=низкий фактор, Q5=высокий)')
    plt.tight_layout()
    plt.show()

    return quint_returns


print('Функция quintile_analysis загружена. Пример:')
print('  quint_df = quintile_analysis(df, factor_col="pe_ratio", return_col="fwd_return", date_col="date")')

---
## 12. Conclusions

Заключительный блок — всегда пиши его явно. Это показывает структурное мышление.

Структура выводов:
1. Что нашёл в данных (EDA)
2. Какая модель лучше и почему
3. Что работает, что нет
4. Что бы исследовал дальше при большем времени

In [ ]:
# === ФИНАЛЬНЫЙ SUMMARY ===

print('=' * 55)
print('SUMMARY')
print('=' * 55)

print('\n1. DATA')
print(f'   Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(f'   Missing: {df.isnull().mean().mean()*100:.1f}% avg across columns')

print('\n2. MODELS — Test RMSE (lower is better)')
for _, row in results_df.iterrows():
    marker = '  <-- BEST' if row['RMSE_test'] == results_df['RMSE_test'].min() else ''
    print(f'   {row.name:25s}: RMSE={row["RMSE_test"]:.4f}, R²={row["R2_test"]:.4f}{marker}')

best_model_name = results_df['RMSE_test'].idxmin()
print(f'\n3. BEST MODEL: {best_model_name}')

print('\n4. NEXT STEPS (if more time):')
print('   - Hyperparameter tuning (GridSearch / Optuna)')
print('   - More feature engineering (interactions, lags)')
print('   - Ensemble / stacking')
print('   - Error analysis: where does the model fail?')
print('   - Residual analysis for regression assumptions')